# 代码生成器

需求：使用 Frontier 模型，从 Python 代码生成高性能的 C++ 代码



<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">提醒：执行 C++ 代码是可选的</h2>
            <span style="color:#f71;">作为替代，你可以在昨天给出的网站上运行</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h1 style="color:#900;">重要提示</h1>
            <span style="color:#900;">
            在本实验中，我在 Ollama 上使用免费的开源模型。我也通过 Groq 和 OpenRouter 使用付费的开源模型。只挑选你想用的模型即可！
            </span>
        </td>
    </tr>
</table>

In [ ]:
# 导入

# 导入标准库 os（操作系统相关，用来读环境变量 Environment Variables）
import os
import io
import sys
# 从 dotenv 导入 load_dotenv：把 .env 文件里的密钥读进环境变量，避免把密钥写进代码
from dotenv import load_dotenv
# 从 openai 导入 OpenAI 客户端类：用它调用 Chat Completions 等 API（Application Programming Interface）
from openai import OpenAI
# 导入 Gradio：快速搭建可交互的 Web 演示界面（聊天框、按钮等）
import gradio as gr
# 导入 subprocess：在 Python 里启动外部命令（如编译器）
import subprocess


In [ ]:
# 加载 .env 文件：把 API Key 等密钥读入进程环境（override=True 表示覆盖已有同名变量）
load_dotenv(override=True)
# 用 os.getenv 读取环境变量里的密钥；找不到时返回 None
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
grok_api_key = os.getenv('GROK_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set (and this is optional)")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if grok_api_key:
    print(f"Grok API Key exists and begins {grok_api_key[:4]}")
else:
    print("Grok API Key not set (and this is optional)")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:6]}")
else:
    print("OpenRouter API Key not set (and this is optional)")



In [ ]:
# 连接到客户端库

# 创建 OpenAI 客户端；不传参时默认读环境变量里的 API Key
openai = OpenAI()

anthropic_url = "https://api.anthropic.com/v1/"
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
grok_url = "https://api.x.ai/v1"
groq_url = "https://api.groq.com/openai/v1"
ollama_url = "http://localhost:11434/v1"
openrouter_url = "https://openrouter.ai/api/v1"

# 用 OpenAI 兼容接口连接其它厂商：关键指定 base_url（服务地址）和 api_key
anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)
grok = OpenAI(api_key=grok_api_key, base_url=grok_url)
groq = OpenAI(api_key=groq_api_key, base_url=groq_url)
ollama = OpenAI(api_key="ollama", base_url=ollama_url)
openrouter = OpenAI(api_key=openrouter_api_key, base_url=openrouter_url)


In [ ]:
# 界面可选的模型列表
models = ["gpt-5", "claude-sonnet-4-5-20250929", "grok-4", "gemini-2.5-pro", "qwen2.5-coder", "deepseek-coder-v2", "gpt-oss:20b", "qwen/qwen3-coder-30b-a3b-instruct", "openai/gpt-oss-120b", ]

# 模型名 → 客户端 的映射表，方便按下拉选项切换厂商
clients = {"gpt-5": openai, "claude-sonnet-4-5-20250929": anthropic, "grok-4": grok, "gemini-2.5-pro": gemini, "openai/gpt-oss-120b": groq, "qwen2.5-coder": ollama, "deepseek-coder-v2": ollama, "gpt-oss:20b": ollama, "qwen/qwen3-coder-30b-a3b-instruct": openrouter}

# 想把成本压到极低？用你选择的模型替换这里，可参考昨天的示例

In [ ]:
# 导入 system_info：读取本机 CPU/编译器等信息，方便后面生成优化代码
from system_info import retrieve_system_info

system_info = retrieve_system_info()
system_info

## 用昨天的命令覆盖这里

或者像昨天一样直接使用网站：

 https://www.programiz.com/cpp-programming/online-compiler/

In [ ]:
# 准备编译命令参数列表（稍后交给 subprocess 执行）
compile_command = ["clang++", "-std=c++17", "-Ofast", "-mcpu=native", "-flto=thin", "-fvisibility=hidden", "-DNDEBUG", "main.cpp", "-o", "main"]
run_command = ["./main"]


## 接下来，进入主要任务

In [ ]:
# 系统提示词（system prompt）：给模型设定角色与规则，通常用户看不到
system_prompt = """
Your task is to convert Python code into high performance C++ code.
Respond only with C++ code. Do not provide any explanation other than occasional comments.
The C++ response needs to produce an identical output in the fastest possible time.
"""

# 根据 Website 对象生成「请摘要此网页」的用户提示词
def user_prompt_for(python):
    return f"""
Port this Python code to C++ with the fastest possible implementation that produces identical output in the least time.
The system information is:
{system_info}
Your response will be written to a file called main.cpp and then compiled and executed; the compilation command is:
{compile_command}
Respond only with C++ code.
Python code to port:

```python
{python}
```
"""

In [ ]:
# 把系统提示词 + 用户内容打包成 API 需要的 messages 结构
def messages_for(python):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(python)}
    ]
 

In [ ]:
# 把模型生成的代码写入本地源文件，供后续编译运行
def write_output(cpp):
    # 打开文件写入；with 结束时自动关闭文件句柄
    with open("main.cpp", "w") as f:
        f.write(cpp)

In [ ]:
# 把 Python 代码「移植」成高性能目标语言：调模型生成代码并写入文件
def port(model, python):
    client = clients[model]
    openai_reasoning_models = {"gpt-5"}
    # reasoning_effort：部分推理模型可调的「思考强度」（如 high），影响耗时与质量
    reasoning_effort = "high" if model in openai_reasoning_models else None
    # 调用 chat.completions.create：向大模型发一次对话请求并拿回复（同步、等全部生成完）
    response = client.chat.completions.create(model=model, messages=messages_for(python), reasoning_effort=reasoning_effort)
    # 从响应里取出第一条候选的 message.content（模型生成的文本）
    reply = response.choices[0].message.content
    # 去掉模型回复里的 Markdown 代码围栏（```），只保留纯代码
    reply = reply.replace('```cpp','').replace('```','')
    write_output(reply)
    return reply

In [ ]:
# 待移植的 Python 示例代码（字符串形式保存，稍后发给模型）
pi = """
import time

def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations+1):
        j = i * param1 - param2
        result -= (1/j)
        j = i * param1 + param2
        result += (1/j)
    return result

start_time = time.time()
result = calculate(200_000_000, 4, 1) * 4
end_time = time.time()

print(f"Result: {result:.12f}")
print(f"Execution Time: {(end_time - start_time):.6f} seconds")
"""

In [ ]:
# 在受控环境里执行 Python 片段，方便和编译后的结果对比耗时
def run_python(code):
    # 准备 exec 的全局命名空间；限制内建可减少意外副作用（演示用）
    globals_dict = {"__builtins__": __builtins__}

    buffer = io.StringIO()
    old_stdout = sys.stdout
    sys.stdout = buffer

    # try/except：尝试执行，出错时进入异常处理，避免整个笔记本崩溃
    try:
        # exec：把字符串当 Python 代码执行（仅用于可信的课程示例）
        exec(code, globals_dict)
        output = buffer.getvalue()
    # 捕获异常：打印或返回友好错误信息
    except Exception as e:
        output = f"Error: {e}"
    finally:
        sys.stdout = old_stdout

    return output

In [ ]:
# 编译刚生成的代码并运行，测量原生性能
def compile_and_run():
    # try/except：尝试执行，出错时进入异常处理，避免整个笔记本崩溃
    try:
        # subprocess.run：执行外部命令；check=True 表示失败就抛异常
        subprocess.run(compile_command, check=True, text=True, capture_output=True)
        print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)
        print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)
        print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)
    # 捕获异常：打印或返回友好错误信息
    except subprocess.CalledProcessError as e:
        print(f"An error occurred:\n{e.stderr}")

In [ ]:
# 用 Gradio Blocks 自定义多组件界面（输入框、按钮、输出区可自由排布）
with gr.Blocks() as ui:
    with gr.Row():
        python = gr.Textbox(label="Python code:", lines=28, value=pi)
        cpp = gr.Textbox(label="C++ code:", lines=28)
    with gr.Row():
        model = gr.Dropdown(models, label="Select model", value=models[0])
        convert = gr.Button("Convert code")

    # 把按钮 click 事件绑到处理函数：inputs 读入，outputs 写回界面
    convert.click(port, inputs=[model, python], outputs=[cpp])

# launch：启动本地 Web 服务并打开演示页面
ui.launch(inbrowser=True)

In [ ]:
compile_and_run()

Qwen 2.5 Coder: Fail  
DeepSeek Coder v2: 0.114050084  
OpenAI gpt-oss 20B: 0.080438  
Qwen 30B: 0.113734  
OpenAI gpt-oss 120B: 1.407383





在 Ed 的实验中，性能加速结果为：

第 9 名：Qwen 2.5 Coder: Fail  
第 8 名：OpenAI GPT-OSS 120B: 14X speedup    
第 7 名：DeepSeek Coder v2: 168X speedup  
第 6 名：Qwen3 Coder 30B: 168X speedup   
第 5 名：Claude Sonnet 4.5: 184X speedup   
第 4 名：GPT-5: 233X speedup  
**第 3 名：oss-20B: 238X speedup**  
第 2 名：Grok 4: 1060X speedup  
第 1 名：Gemini 2.5 Pro: 1440X speedup  